In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, IntegerType, DateType, TimestampType, FloatType, DoubleType
from pyspark.sql import Row

catalog_name = "ecommerce"

In [0]:
df_customers_clean = spark.read.table(f"{catalog_name}.silver.slv_customers_clean")
df_countries_clean = spark.read.table(f"{catalog_name}.silver.slv_countries_clean")

display(df_customers_clean)

In [0]:
india_region = {
    "MH": "West", "GJ": "West", "RJ": "West",
    "KA": "South", "TN": "South", "AP": "South", "KL": "South",
    "UP": "North", "WB": "North", "DL": "North"
}

australia_region = {
    "VIC": "Southeast", "WA": "West", "NSW": "East", "QLD": "Northeast"
}

uk_region = {
    "ENG": "England", "WLS": "Wales", "NIR": "Northern Ireland", "SCT": "Scotland"
}

us_region = {
    "MA": "Northeast", "FL": "South", "NJ": "Northeast", "CA": "West",
    "NY": "Northeast", "TX": "South"
}

uae_region = {
    "AUH": "A"
}

singapore_region = {
    "SG": "Singapore"
}

canada_region = {
    "BC": "West", "AB": "West", "ON": "East", "QC": "East", "NS": "East", "IL": "Other"
}

country_state_map = {
    "India": india_region,
    "Australia": australia_region,
    "United Kingdom": uk_region,
    "United States": us_region,
    "United Arab Emirates": uae_region,
    "Singapore": singapore_region,
    "Canada": canada_region
}

In [0]:
country_state_map

In [0]:
rows = []
for country, states in country_state_map.items():
    for state_code, region in states.items():
        rows.append(Row(country_name=country, state=state_code, region=region))
rows[:10]

In [0]:
df_region_mapping = spark.createDataFrame(rows)

df_region_mapping.show(truncate = False)

In [0]:
gold_customers_df = df_customers_clean.join(df_region_mapping, on = ['country_name', 'state'], how = 'left')

gold_customers_df = gold_customers_df.fillna({'region':'Other'})

display(gold_customers_df.limit(30))

In [0]:
gold_customers_df.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog_name}.gold.gld_dim_customers")
